In [30]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

In [31]:
df_labeled = pd.read_csv("Data download/data/filings_labeled.csv")
print(df_labeled.columns.tolist())

['ticker', 'cik', 'accessionNumber', 'filingDate', 'filingDatetime', 'event_date', 'event_date_m1', 'event_date_p1', 'price_m1', 'price_0', 'price_p1', 'spy_price_m1', 'spy_price_0', 'spy_price_p1', 'mm_alpha', 'mm_beta', 'mm_flag', 'ret_stock', 'ret_spy', 'expected_ret', 'car_0_1', 'car_0_1_mktadj', 'downside', 'event_date_m2', 'price_m2', 'spy_price_m2']


In [32]:
DATA_DIR = Path("Data download/data")


In [33]:
# ── CAR[0,0] — same day only ──────────────────────────────────────────────────
df_labeled['ar_0']    = df_labeled['ret_stock'] - (df_labeled['mm_alpha'] + df_labeled['mm_beta'] * df_labeled['ret_spy'])
df_labeled['car_0_0'] = df_labeled['ar_0']

# ── CAR[−1,+1] — three-day window — CORRECTED ────────────────────────────────
# Day -1 return: price moves from price_m2 (day -2 close) to price_m1 (day -1 close)
df_labeled['ret_stock_m1'] = (df_labeled['price_m1'] - df_labeled['price_m2']) / df_labeled['price_m2']
df_labeled['ret_spy_m1']   = (df_labeled['spy_price_m1'] - df_labeled['spy_price_m2']) / df_labeled['spy_price_m2']
df_labeled['ar_m1']        = df_labeled['ret_stock_m1'] - (df_labeled['mm_alpha'] + df_labeled['mm_beta'] * df_labeled['ret_spy_m1'])
df_labeled['ar_1']         = df_labeled['car_0_1'] - df_labeled['ar_0']
df_labeled['car_m1_1']     = df_labeled['ar_m1'] + df_labeled['ar_0'] + df_labeled['ar_1']

# ── Market-adjusted versions ──────────────────────────────────────────────────
df_labeled['ret_stock_1']     = (df_labeled['price_p1'] - df_labeled['price_0']) / df_labeled['price_0']
df_labeled['ret_spy_1']       = (df_labeled['spy_price_p1'] - df_labeled['spy_price_0']) / df_labeled['spy_price_0']
df_labeled['car_0_0_mktadj']  = df_labeled['ret_stock'] - df_labeled['ret_spy']
df_labeled['car_m1_1_mktadj'] = (
    (df_labeled['ret_stock_m1'] - df_labeled['ret_spy_m1']) +
    (df_labeled['ret_stock']    - df_labeled['ret_spy'])    +
    (df_labeled['ret_stock_1']  - df_labeled['ret_spy_1'])
)

# ── Sanity check ──────────────────────────────────────────────────────────────
print("New columns computed:")
for col in ['car_0_0', 'car_0_1', 'car_m1_1']:
    s = df_labeled[col].dropna()
    print(f"  {col}: mean={s.mean():.4f}, std={s.std():.4f}, "
          f"min={s.min():.4f}, max={s.max():.4f}")

print(f"\nExpected: car_m1_1 std should be noticeably larger than car_0_1 std")
print(f"Ratio car_m1_1/car_0_1 std: {df_labeled['car_m1_1'].std() / df_labeled['car_0_1'].std():.3f}")
print("(Expected ~1.3–1.6. If ~1.0 the bug is still present.)")

New columns computed:
  car_0_0: mean=-0.0002, std=0.0548, min=-0.4529, max=0.4386
  car_0_1: mean=0.0002, std=0.0740, min=-0.5057, max=0.4301
  car_m1_1: mean=0.0006, std=0.0755, min=-0.5080, max=0.4137

Expected: car_m1_1 std should be noticeably larger than car_0_1 std
Ratio car_m1_1/car_0_1 std: 1.020
(Expected ~1.3–1.6. If ~1.0 the bug is still present.)


In [34]:
# Check the implied day -1 AR variance
import numpy as np

std_01  = df_labeled['car_0_1'].std()
std_m11 = df_labeled['car_m1_1'].std()

# var(car_m1_1) = var(car_0_1) + var(AR[-1])  (if uncorrelated)
implied_ar_m1_std = np.sqrt(std_m11**2 - std_01**2)
print(f"Implied AR[-1] std : {implied_ar_m1_std:.4f} ({implied_ar_m1_std*100:.2f}%)")
print(f"AR[0] std (car_0_0): {df_labeled['car_0_0'].std():.4f}")

# Day -1 should look like a normal daily return (~1.5-2% for S&P 500 large caps)
# Day 0 carries the full event-day shock, hence ~5.5% std

Implied AR[-1] std : 0.0147 (1.47%)
AR[0] std (car_0_0): 0.0548


In [35]:
# ── Load all three checkpoints ────────────────────────────────────────────────
df_lm      = pd.read_csv(DATA_DIR / "lm_checkpoint.csv")
df_finbert = pd.read_csv(DATA_DIR / "finbert_checkpoint.csv")
df_gpt     = pd.read_csv(DATA_DIR / "gpt_checkpoint.csv").dropna(subset=['gpt_score'])

# ── Build the held-out evaluation set ─────────────────────────────────────────
# Exclude every filing with prior contact with the scoring process:
#   (1) the 400 filings scored during an earlier prompt-development run, and
#   (2) the four labeled examples reproduced in the Appendix A prompt.
# All 400 early-run filings are nested within the 2,000-filing subsample.
# This matches the held-out evaluation set used in 06_results.
EARLY_RUN_PATH = DATA_DIR / "gpt_checkpoint400.csv"  
early_run_acc  = set(pd.read_csv(EARLY_RUN_PATH)['accessionNumber'])

FEWSHOT = {
    "0001071739-25-000128",  # CNC
    "0000798354-25-000168",  # FISV
    "0001104659-24-094354",  # DG
    "0001069183-24-000055",  # AXON (not in subsample)
}

df_gpt = df_gpt[~df_gpt['accessionNumber'].isin(early_run_acc | FEWSHOT)].copy()

# ── GPT-scored held-out evaluation set ───────────────────────────────────────
gpt_eval = df_gpt.copy()

# ── Filter LM and FinBERT to the exact same filings as GPT ───────────────────
gpt_eval_acc = set(gpt_eval['accessionNumber'])
lm_eval      = df_lm[df_lm['accessionNumber'].isin(gpt_eval_acc)].copy()
finbert_eval = df_finbert[df_finbert['accessionNumber'].isin(gpt_eval_acc)].copy()

print(f"GPT eval     : {len(gpt_eval)}")    # expect 1597
print(f"LM eval      : {len(lm_eval)}")     # expect 1597
print(f"FinBERT eval : {len(finbert_eval)}")  # expect 1597

GPT eval     : 1597
LM eval      : 1597
FinBERT eval : 1597


In [36]:
# Merge alternative CARs into evaluation subsamples
df_rob = df_labeled[['accessionNumber', 'car_0_0', 'car_m1_1', 
                       'car_0_0_mktadj', 'car_m1_1_mktadj']]

lm_rob      = lm_eval.merge(df_rob, on='accessionNumber')
finbert_rob = finbert_eval.merge(df_rob, on='accessionNumber')
gpt_rob     = gpt_eval.merge(df_rob, on='accessionNumber')

print("=== Robustness Check 1: Alternative Event Windows ===")
print("(Primary metric: AUC on evaluation subsample, N=2,000)\n")

windows = {
    'CAR[0,0]'   : 'car_0_0',
    'CAR[0,+1]'  : 'car_0_1',   # primary — for reference
    'CAR[-1,+1]' : 'car_m1_1',
}

print(f"{'Window':<14} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 50)

for window_name, car_col in windows.items():
    # Recompute downside label at 25th percentile for each window
    threshold    = df_labeled[car_col].quantile(0.25)
    df_labeled[f'downside_{car_col}'] = (df_labeled[car_col] < threshold).astype(int)

    lm_rob      = lm_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')
    finbert_rob = finbert_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')
    gpt_rob     = gpt_eval.merge(df_labeled[['accessionNumber', car_col, f'downside_{car_col}']], on='accessionNumber')

    auc_lm  = roc_auc_score(lm_rob[f'downside_{car_col}'],      lm_rob['lm_neg_prop'])
    auc_fb  = roc_auc_score(finbert_rob[f'downside_{car_col}'], finbert_rob['finbert_neg_mean'])
    auc_gpt = roc_auc_score(gpt_rob[f'downside_{car_col}'],    -gpt_rob['gpt_score'])

    primary = " ← primary" if car_col == 'car_0_1' else ""
    print(f"{window_name:<14} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}{primary}")

=== Robustness Check 1: Alternative Event Windows ===
(Primary metric: AUC on evaluation subsample, N=2,000)

Window             LM AUC  FinBERT AUC    GPT AUC
──────────────────────────────────────────────────
CAR[0,0]           0.4811       0.4687     0.5006
CAR[0,+1]          0.5165       0.5130     0.5749 ← primary
CAR[-1,+1]         0.5093       0.5245     0.5749


In [37]:
print("=== Robustness Check 2: Market-Adjusted Returns ===\n")

# Compute market-adjusted downside label
threshold_mktadj = df_labeled['car_0_1_mktadj'].quantile(0.25)
df_labeled['downside_mktadj'] = (df_labeled['car_0_1_mktadj'] < threshold_mktadj).astype(int)

print(f"{'Return model':<28} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 64)

# Primary: market model — use existing downside column already in evaluation subsamples
auc_lm  = roc_auc_score(lm_eval['downside'],      lm_eval['lm_neg_prop'])
auc_fb  = roc_auc_score(finbert_eval['downside'],  finbert_eval['finbert_neg_mean'])
auc_gpt = roc_auc_score(gpt_eval['downside'],     -gpt_eval['gpt_score'])
print(f"{'CAR[0,+1] market model':<28} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f} ← primary")

# Robustness: market-adjusted — merge only the new mktadj downside label
mktadj_labels = df_labeled[['accessionNumber', 'downside_mktadj']]
lm_rob      = lm_eval.merge(mktadj_labels, on='accessionNumber')
finbert_rob = finbert_eval.merge(mktadj_labels, on='accessionNumber')
gpt_rob     = gpt_eval.merge(mktadj_labels, on='accessionNumber')

auc_lm  = roc_auc_score(lm_rob['downside_mktadj'],      lm_rob['lm_neg_prop'])
auc_fb  = roc_auc_score(finbert_rob['downside_mktadj'],  finbert_rob['finbert_neg_mean'])
auc_gpt = roc_auc_score(gpt_rob['downside_mktadj'],     -gpt_rob['gpt_score'])
print(f"{'CAR[0,+1] market adj':<28} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}")

=== Robustness Check 2: Market-Adjusted Returns ===

Return model                     LM AUC  FinBERT AUC    GPT AUC
────────────────────────────────────────────────────────────────
CAR[0,+1] market model           0.5165       0.5130     0.5749 ← primary
CAR[0,+1] market adj             0.5261       0.5211     0.5791


In [38]:
print("=== Robustness Check 3: Downside Threshold Sensitivity ===\n")

thresholds = {
    'Percentile 25% (−3.90%)'  : df_labeled['car_0_1'].quantile(0.25),   # primary
    'Fixed −1%'                : -0.01,
    'Fixed −3%'                : -0.03,
}

print(f"{'Threshold':<28} {'Downside N':>12} {'LM AUC':>10} {'FinBERT AUC':>12} {'GPT AUC':>10}")
print("─" * 76)

for threshold_name, cutoff in thresholds.items():
    df_labeled['downside_thresh'] = (df_labeled['car_0_1'] < cutoff).astype(int)

    lm_rob      = lm_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')
    finbert_rob = finbert_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')
    gpt_rob     = gpt_eval.merge(df_labeled[['accessionNumber', 'downside_thresh']], on='accessionNumber')

    n_down = lm_rob['downside_thresh'].sum()

    try:
        auc_lm  = roc_auc_score(lm_rob['downside_thresh'],      lm_rob['lm_neg_prop'])
        auc_fb  = roc_auc_score(finbert_rob['downside_thresh'],  finbert_rob['finbert_neg_mean'])
        auc_gpt = roc_auc_score(gpt_rob['downside_thresh'],     -gpt_rob['gpt_score'])
        primary = " ← primary" if 'Percentile' in threshold_name else ""
        print(f"{threshold_name:<28} {n_down:>12} {auc_lm:>10.4f} {auc_fb:>12.4f} {auc_gpt:>10.4f}{primary}")
    except ValueError as e:
        print(f"{threshold_name:<28} skipped — {e}")

=== Robustness Check 3: Downside Threshold Sensitivity ===

Threshold                      Downside N     LM AUC  FinBERT AUC    GPT AUC
────────────────────────────────────────────────────────────────────────────
Percentile 25% (−3.90%)               797     0.5165       0.5130     0.5749 ← primary
Fixed −1%                             984     0.4942       0.5033     0.5629
Fixed −3%                             856     0.5134       0.5133     0.5681
